## Currency Conversion Tool

In [1]:
# !uv add langchain_huggingface langchain-core requests

In [2]:
import os
import json
import requests
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

from langchain_core.tools import InjectedToolArg
from typing import Annotated

load_dotenv()

hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

### Tool Create

#### Remember 
Hugging Face LLM + LangChain currently cannot auto-chain multiple tools where one depends on the output of another via InjectedToolArg. That’s why your two-tool approach fails.

In [3]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency Conversion Factor between a given base currenct and a target currency and call convert tool for converting base currency into a target currency
    """
    # Where USD is the base currency you want to use
    url = f'https://v6.exchangerate-api.com/v6/cf9f2c5f307e450ac82dffe8/pair/{base_currency}/{target_currency}'

    # Making our request
    response = requests.get(url)
    data = response.json()

    # Your JSON object
    return data["conversion_rate"]

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Given a currency conversion rate (get using get_conversion_factor tool) this function calculate the target currency value from a given base currency value
    """
    return base_currency_value * conversion_rate

In [4]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'EUR'})

0.8525

In [5]:
convert.invoke({"base_currency_value": 10, "conversion_rate": 88.3176})

883.1759999999999

### Tool binding

In [6]:
llm = HuggingFaceEndpoint(
    repo_id = "openai/gpt-oss-20b",
    task="task-generation"
)

chat_llm = ChatHuggingFace(llm=llm)

/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# llm_with_tools = chat_llm.bind_tools([get_conversion_factor, convert])
llm_with_tools = chat_llm.bind_tools([get_conversion_factor, convert])
# llm_with_tools

In [8]:
messages = [HumanMessage("First get me the conversion factor between USD and INR, then convert 100 USD to INR.")]

In [9]:
messages

[HumanMessage(content='First get me the conversion factor between USD and INR, then convert 100 USD to INR.', additional_kwargs={}, response_metadata={})]

In [10]:
ai_message = llm_with_tools.invoke(messages)

In [11]:
ai_message
messages.append(ai_message)
# ai_message.tool_calls

In [12]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_JqfyFqdgAeoxJRgIV7x3fHzP',
  'type': 'tool_call'}]

In [13]:
for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call["name"] == "get_conversion_factor":
        tool_message1 = get_conversion_factor.invoke(tool_call)
        # fetch conversion rate
        # print(json.loads(tool_message1.content)["conversion_rate"])
        conversion_rate = tool_message1.content
        # Append this tool message to message list
        messages.append(tool_message1)

    # Execute the 2nd tool using the conversion rate from tool 1
    if tool_call['name'] == "convert":
        # fetch the current arg
        tool_call["args"]["conversion_rate"] = conversion_rate
        tool_message2 = convert.invoke[tool_call]
        messages.append(tool_message2)

In [14]:
messages

[HumanMessage(content='First get me the conversion factor between USD and INR, then convert 100 USD to INR.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_JqfyFqdgAeoxJRgIV7x3fHzP', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 217, 'total_tokens': 283}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--e9426b6e-c6f3-4ab1-96cc-468f45f01427-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_JqfyFqdgAeoxJRgIV7x3fHzP', 'type': 'tool_call'}], usage_metadata={'input_tokens': 217, 'output_tokens': 66, 'total_tokens': 283}),
 ToolMessage(content='88.3176', name='get_conversion_factor', tool_call_id='

In [15]:
llm_with_tools.invoke(messages).content

''